# Imports

In [28]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


# Load dataset

In [29]:
df = pd.read_csv("/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv", encoding="latin-1")
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

print(df.head())
print("Documents:", len(df))


  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
Documents: 5572


# LabelEncoder

In [30]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])


In [31]:
df

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [ ]:
X = df['text']          # pandas Series
y = df['label'].astype(int)   # pandas Series of ints


# Count ham vs spam

In [18]:
df['label'].value_counts()


label
0    4825
1     747
Name: count, dtype: int64

 the SMS Spam dataset is imbalanced.

# Define all models you want to test

In [39]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    'KNN': KNeighborsClassifier(),
    'LogReg': LogisticRegression(max_iter=1000),
    'RF': RandomForestClassifier()
}


# Create Stratified K‑Fold (for imbalanced data)

In [40]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Loop through each model + each fold

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

for name, model in models.items():
    scores = []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        clf = Pipeline([
            ('tfidf', TfidfVectorizer()),
            ('clf', model)
        ])

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        scores.append(accuracy_score(y_test, y_pred))

    print(name, "mean accuracy:", sum(scores) / len(scores))
    print(classification_report(y_test, y_pred))

KNN mean accuracy: 0.9097268357874906
              precision    recall  f1-score   support

           0       0.91      1.00      0.95       965
           1       1.00      0.36      0.52       149

    accuracy                           0.91      1114
   macro avg       0.95      0.68      0.74      1114
weighted avg       0.92      0.91      0.90      1114

LogReg mean accuracy: 0.9720028016842308
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       0.99      0.81      0.89       149

    accuracy                           0.97      1114
   macro avg       0.98      0.91      0.94      1114
weighted avg       0.97      0.97      0.97      1114

RF mean accuracy: 0.9788221655086907
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       1.00      0.85      0.92       149

    accuracy                           0.98      1114
   macro avg

| Model               |   Accuracy | Spam Precision | Spam Recall | Spam F1 |
| ------------------- | ---------: | -------------: | ----------: | ------: |
| KNN                 | **0.9097** |           1.00 |        0.36 |    0.52 |
| Logistic Regression | **0.9720** |           0.99 |        0.81 |    0.89 |
| Random Forest       | **0.9788** |           1.00 |        0.85 |    0.92 |
